# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ARS-0/flyrank-ml-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing **Lane 4: CTR / Engagement Opportunity Scoring** (provisional — I can change this by Week 4).

In Notebook 01 I already found that CTR collapses sharply by position tier (page_1 ≈0.35 down to
deep ≈0.05), and that content_type also matters *within* the same tier — comparison articles
underperform other types even when ranked similarly. That combination (position explains some of
CTR, but not all of it) is exactly the "expected CTR by tier, then find who's below expectation"
problem this lane is built around. It also connects to something a content team can actually act
on: rewriting a title/meta or improving snippet structure, not something abstract like "predict
Google's algorithm."

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Question:** Which visible pages are under-capturing clicks relative to other pages at the same
position tier, and are worth a human reviewer's time to check first?

**Unit of analysis:** one page (content_id), scored using its trailing performance window.

**Decision this improves:** which pages a content reviewer opens first out of a large backlog,
when they only have time to check a handful this week.

**Action someone could take:** rewrite the page's title/meta description, improve the snippet/schema,
or re-check search intent match — then monitor CTR again after the change.

**Cost of a wrong recommendation:**
- False positive (flagged as underperforming, but it's actually fine): wastes a reviewer's time,
  low cost, but erodes trust in the queue if it happens a lot.
- False negative (a genuinely weak page never gets flagged): a real opportunity is missed, but
  since this feeds a repeatable weekly queue rather than a one-shot decision, it isn't catastrophic —
  it just delays the fix.

**Why data/ML helps at all:** with over a hundred thousand pages, a human can't eyeball everything.
Ranking by "how far below its tier's expected CTR" turns an unmanageable list into a short,
explainable queue — no ML "prediction" required for a first pass, but a learned model can later
account for more factors (intent, content_type, freshness) than a single group-mean baseline can.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:

import pandas as pd

import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/ARS-0/FlyRank-ML-Internship"
REPO_DIR = "FlyRank-ML-Internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check repo path"

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: CTR collapses by position tier (the "cliff")
visible = df[df["impressions_90d"] >= 100]
ctr_by_pos = visible.groupby("position_tier")["ctr"].mean().sort_values(ascending=False)
print("Mean CTR by position tier (impressions >= 100):")
print(ctr_by_pos.round(4).to_string())

# Number 2: content_type matters within a tier, not just position
ctr_by_type_pos = (
    visible.groupby(["position_tier", "content_type"])["ctr"]
    .agg(["mean", "count"])
    .sort_values("mean")
)
print("\nCTR by content_type within tier (filtered to impressions >= 100 to reduce noise):")
print(ctr_by_type_pos.round(4).to_string())

# Number 3: how many pages this lane's queue would actually be working with
n_visible = len(visible)
n_total = len(df)
print(f"\n{n_visible} of {n_total} pages ({n_visible/n_total:.1%}) have enough impressions "
      f"(>=100) to trust a CTR comparison — this is roughly the review-eligible pool size.")


Working dir: /content/YOUR-REPO-NAME/YOUR-REPO-NAME/YOUR-REPO-NAME/FlyRank-ML-Internship
Mean CTR by position tier (impressions >= 100):
position_tier
page_1      0.3548
top_3       0.3341
striking    0.2558
page_3_5    0.1424
deep        0.0554

CTR by content_type within tier (filtered to impressions >= 100 to reduce noise):
                                    mean  count
position_tier content_type                     
top_3         comparison article  0.0000      1
deep          feedly article      0.0440     10
              keyword article     0.0555    869
page_3_5      comparison article  0.0940     63
page_1        comparison article  0.1412    226
page_3_5      keyword article     0.1427   5954
striking      comparison article  0.1474     76
page_3_5      feedly article      0.1656     41
striking      keyword article     0.2559   5752
top_3         keyword article     0.3104    527
page_1        keyword article     0.3458   8186
striking      feedly article      0.3580     75

**What these numbers show:** CTR ranges from ~0.35 at page_1 down to ~0.05 "deep" — a ~7x spread
by position alone, so position must be controlled for before flagging anything as "underperforming."
Within the same tier, content_type still creates meaningful gaps (seen earlier: comparison article
underperforms keyword article at top_3, page_3_5, and striking). And only 73.4% of pages have enough
impressions to trust a CTR comparison at all — this sets the realistic size of the queue, not
hundreds of thousands of pages, but a filtered, trustworthy subset.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**I can claim:** these are *observed, directional* patterns in this anonymized starter slice —
CTR is associated with position tier and, within a tier, with content_type. A ranked queue built
from this can help a reviewer prioritize.

**I cannot claim:** that any specific content_type "causes" lower CTR, that fixing a flagged page
will definitely recover clicks (that needs a real before/after experiment, not this dataset), or
that this reflects how Google's ranking algorithm works. Some of the observed gaps (like the
extreme feedly-article values I saw) come from very small sample sizes and shouldn't be trusted
without a minimum-volume filter. My starter-dataset numbers also come from one 30,000-row
anonymized sample — they set direction, not a guaranteed result on the full warehouse.

## 5. Self-check

- [x] Picked a lane (Lane 4, provisional) and explained why
- [x] Named the decision (which pages to review first) and the action (title/meta/snippet rewrite)
- [x] Named the cost of a wrong call (reviewer time vs. missed opportunity)
- [x] Showed 2-3 real numbers from the starter dataset (CTR-by-tier cliff, content_type gap,
      eligible-page count)
- [x] Used observed/directional language, not causal claims

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.